In [ ]:
gt_json = "/Users/delmedigo/Dev/FineTree/data/annotations/P907062-00.json"
pred_1 = "/Users/delmedigo/Dev/FineTree/benchmark/Qwen3.5-27B-FineTree-V2.1/pred_1.txt"
pred_2 = "/Users/delmedigo/Dev/FineTree/benchmark/Qwen3.5-27B-FineTree-V2.1/pred_2.txt"
pred_3 = "/Users/delmedigo/Dev/FineTree/benchmark/Qwen3.5-27B-FineTree-V2.1/pred_3.txt"

In [ ]:
import json

with open(gt_json, "r") as f:
    data = json.load(f)

gt_1 = data.get("pages")[3]
gt_2 = data.get("pages")[1]
gt_3 = data.get("pages")[5]

gts = [gt_1, gt_2, gt_3]
pops = ["annotation_note", "annotation_status"]
for gt in gts:
    gt.pop("image")
    for _pop in pops:
        gt.get("meta").pop(_pop)

with open(pred_1, "r") as f:
    pred_1_dict = json.loads(f.read())

with open(pred_2, "r") as f:
    pred_2_dict = json.loads(f.read())

with open(pred_3, "r") as f:
    pred_3_dict = json.loads(f.read())

In [ ]:
from typing import Dict, List, Any, Union
import pydash


## Meta Data Checks

def get_value_by_path(d: Dict[str, Any], path: List[Union[str, int]]) -> Any:
    parts = []

    for p in path:
        if isinstance(p, int):
            parts[-1] = f"{parts[-1]}[{p}]"
        else:
            parts.append(p)

    return pydash.get(d, ".".join(parts))


def hard_eq_check(dict_1: Dict, dict_2: Dict, path: List[str]) -> bool:
    return get_value_by_path(dict_1, path) == get_value_by_path(dict_2, path)


def hard_len_check(dict_1: Dict, dict_2: Dict, path: List[str]) -> bool:
    v1 = get_value_by_path(dict_1, path)
    v2 = get_value_by_path(dict_2, path)

    if v1 is None or v2 is None:
        return False

    return len(v1) == len(v2)


paths = [
    ["meta", "entity_name"],
    ["meta", "page_num"],
    ["meta", "page_type"],
    ["meta", "statement_type"],
    ["meta", "title"],
]

count = 0
success = 0
for path in paths:
    count += 1
    check = hard_eq_check(pred_3_dict, gt_3, path)
    if check:
        success += 1
    print(path[-1], ":", check)

print("meta score", ":",f"{success / count * 100}%")

In [ ]:
from pprint import pprint

pprint(pred_3_dict["meta"], indent=2, width=100, sort_dicts=False)

In [ ]:
from pprint import pprint

pprint(gt_3["meta"], indent=2, width=100, sort_dicts=False)

In [69]:
import re
from difflib import SequenceMatcher
from statistics import mean
from typing import Any, Dict, List, Union

import pydash

Path = List[Union[str, int]]

META_SCORING = {
    ("meta", "entity_name"): "soft",
    ("meta", "page_num"): "hard",
    ("meta", "page_type"): "hard",
    ("meta", "statement_type"): "hard",
    ("meta", "title"): "soft",
}


def tokenize(s: str) -> List[str]:
    """
    Split a string into lowercase word-like tokens.

    This is used for fuzzy matching of text fields such as
    entity names and titles.

    Args:
        s: Input string.

    Returns:
        List of lowercase tokens.
    """
    return re.findall(r"\w+", s.lower())


def char_similarity(a: str, b: str) -> float:
    """
    Compute character-level similarity between two strings.

    Uses difflib.SequenceMatcher ratio, which is useful for
    OCR-like mistakes such as one-letter differences.

    Args:
        a: First string.
        b: Second string.

    Returns:
        Float in [0, 1].
    """
    return SequenceMatcher(None, a, b).ratio()


def fuzzy_token_overlap_score(a: str, b: str) -> float:
    """
    Compute a soft similarity score between two strings by matching
    tokens fuzzily instead of requiring exact token equality.

    Each token in `a` is matched to the most similar unused token in `b`.
    The final score is the sum of best token similarities divided by the
    maximum token count across both strings.

    This is tolerant to OCR noise and minor spelling mistakes.

    Args:
        a: First string.
        b: Second string.

    Returns:
        Float in [0, 1].
    """
    tokens_a = tokenize(a)
    tokens_b = tokenize(b)

    if not tokens_a or not tokens_b:
        return 0.0

    scores = []
    used_j = set()

    for tok_a in tokens_a:
        best_score = 0.0
        best_j = None

        for j, tok_b in enumerate(tokens_b):
            if j in used_j:
                continue

            score = char_similarity(tok_a, tok_b)
            if score > best_score:
                best_score = score
                best_j = j

        if best_j is not None:
            used_j.add(best_j)
            scores.append(best_score)

    denom = max(len(tokens_a), len(tokens_b))
    return sum(scores) / denom if denom else 0.0


def get_value_by_path(d: Dict[str, Any], path: Path) -> Any:
    """
    Retrieve a nested value from a dictionary using a mixed path of
    string keys and integer list indices.

    Example:
        ["meta", "title"]
        ["facts", 0, "value"]

    Args:
        d: Input nested dictionary.
        path: Path as list of keys / indices.

    Returns:
        The value at the given path, or None if not found.

    Raises:
        ValueError: If the path starts with an integer.
    """
    parts: List[str] = []

    for p in path:
        if isinstance(p, int):
            if not parts:
                raise ValueError("Path cannot start with int for this path format")
            parts[-1] = f"{parts[-1]}[{p}]"
        else:
            parts.append(str(p))

    return pydash.get(d, ".".join(parts))


def path_to_str(path: Path) -> str:
    """
    Convert a path list into a readable dotted string.

    Example:
        ["meta", "title"] -> "meta.title"
        ["facts", 0, "value"] -> "facts[0].value"

    Args:
        path: Path list.

    Returns:
        Readable path string.
    """
    out: List[str] = []

    for p in path:
        if isinstance(p, int):
            if not out:
                out.append(f"[{p}]")
            else:
                out[-1] = f"{out[-1]}[{p}]"
        else:
            out.append(str(p))

    return ".".join(out)


def hard_eq_score(dict_1: Dict[str, Any], dict_2: Dict[str, Any], path: Path) -> float:
    """
    Check whether two values at a given path are exactly equal.

    Args:
        dict_1: First dictionary.
        dict_2: Second dictionary.
        path: Path to compare.

    Returns:
        1.0 if equal, else 0.0.
    """
    return float(get_value_by_path(dict_1, path) == get_value_by_path(dict_2, path))


def soft_eq_score(dict_1: Dict[str, Any], dict_2: Dict[str, Any], path: Path) -> float:
    """
    Compute a soft similarity score between two values at a given path.

    If the values are exactly equal, returns 1.0.
    If both are strings, uses fuzzy token overlap score.
    Otherwise returns 0.0.

    Args:
        dict_1: First dictionary.
        dict_2: Second dictionary.
        path: Path to compare.

    Returns:
        Float in [0, 1].
    """
    v1 = get_value_by_path(dict_1, path)
    v2 = get_value_by_path(dict_2, path)

    if v1 == v2:
        return 1.0

    if not isinstance(v1, str) or not isinstance(v2, str):
        return 0.0

    return fuzzy_token_overlap_score(v1, v2)


def hard_len_score(dict_1: Dict[str, Any], dict_2: Dict[str, Any], path: Path) -> float:
    """
    Compare the lengths of two values at a given path.

    Intended mainly for comparing list lengths, such as number of facts.

    Args:
        dict_1: First dictionary.
        dict_2: Second dictionary.
        path: Path to compare.

    Returns:
        1.0 if lengths are equal, else 0.0.
    """
    v1 = get_value_by_path(dict_1, path)
    v2 = get_value_by_path(dict_2, path)

    if v1 is None or v2 is None:
        return 0.0

    try:
        return float(len(v1) == len(v2))
    except TypeError:
        return 0.0


def build_meta_field_report(pred_page: Dict[str, Any], true_page: Dict[str, Any]) -> Dict[str, Any]:
    """
    Build a detailed per-field report for page meta comparison.

    For each configured meta field, includes:
    - scoring method
    - predicted value
    - true value
    - hard score
    - soft score
    - used score
    - exact match flag
    - error label

    Args:
        pred_page: Predicted page dictionary.
        true_page: Ground-truth page dictionary.

    Returns:
        Dictionary keyed by field name.
    """
    field_reports = {}

    for path_tuple, method in META_SCORING.items():
        path = list(path_tuple)
        field_name = path[-1]

        pred_value = get_value_by_path(pred_page, path)
        true_value = get_value_by_path(true_page, path)

        hard_score = hard_eq_score(pred_page, true_page, path)
        soft_score = soft_eq_score(pred_page, true_page, path)
        used_score = soft_score if method == "soft" else hard_score

        field_reports[field_name] = {
            "path": path_to_str(path),
            "method": method,
            "pred_value": pred_value,
            "true_value": true_value,
            "hard_score": hard_score,
            "soft_score": soft_score,
            "used_score": used_score,
            "is_exact_match": pred_value == true_value,
            "error": None if pred_value == true_value else "value_mismatch",
        }

    return field_reports


def evaluate_meta_hard(pred_page: Dict[str, Any], true_page: Dict[str, Any]) -> float:
    """
    Compute strict meta score where all configured meta fields are compared
    using exact equality.

    Args:
        pred_page: Predicted page dictionary.
        true_page: Ground-truth page dictionary.

    Returns:
        Mean hard score across configured meta fields.
    """
    paths = [list(path) for path in META_SCORING.keys()]
    scores = [hard_eq_score(pred_page, true_page, path) for path in paths]
    return mean(scores) if scores else 0.0


def evaluate_meta_soft(pred_page: Dict[str, Any], true_page: Dict[str, Any]) -> float:
    """
    Compute relaxed meta score using mixed scoring rules:
    - soft matching for fuzzy text fields
    - hard matching for categorical / exact fields

    Args:
        pred_page: Predicted page dictionary.
        true_page: Ground-truth page dictionary.

    Returns:
        Mean mixed score across configured meta fields.
    """
    field_reports = build_meta_field_report(pred_page, true_page)
    scores = [info["used_score"] for info in field_reports.values()]
    return mean(scores) if scores else 0.0


def evaluate_meta(pred_page: Dict[str, Any], true_page: Dict[str, Any]) -> Dict[str, Any]:
    """
    Evaluate page-level meta information.

    Returns:
    - hard_score: exact strict score over all meta fields
    - soft_score: mixed score using configured hard/soft rules
    - field_scores: compact per-field scores
    - field_reports: detailed per-field diagnostics

    Args:
        pred_page: Predicted page dictionary.
        true_page: Ground-truth page dictionary.

    Returns:
        Meta evaluation report.
    """
    field_reports = build_meta_field_report(pred_page, true_page)

    return {
        "hard_score": evaluate_meta_hard(pred_page, true_page),
        "soft_score": evaluate_meta_soft(pred_page, true_page),
        "field_scores": {
            field_name: {
                "method": info["method"],
                "hard_score": info["hard_score"],
                "soft_score": info["soft_score"],
                "used_score": info["used_score"],
            }
            for field_name, info in field_reports.items()
        },
        "field_reports": field_reports,
    }


def evaluate_facts(pred_page: Dict[str, Any], true_page: Dict[str, Any]) -> Dict[str, Any]:
    """
    Evaluate facts at page level.

    Current implementation only compares fact counts, not fact content.

    Args:
        pred_page: Predicted page dictionary.
        true_page: Ground-truth page dictionary.

    Returns:
        Dictionary with fact-count diagnostics.
    """
    pred_facts = get_value_by_path(pred_page, ["facts"])
    true_facts = get_value_by_path(true_page, ["facts"])

    pred_len = len(pred_facts) if isinstance(pred_facts, list) else None
    true_len = len(true_facts) if isinstance(true_facts, list) else None
    facts_len_score = hard_len_score(pred_page, true_page, ["facts"])

    return {
        "facts_len_score": facts_len_score,
        "pred_fact_count": pred_len,
        "true_fact_count": true_len,
        "count_difference": None if pred_len is None or true_len is None else pred_len - true_len,
        "is_exact_fact_count_match": pred_len == true_len,
        "error": None if pred_len == true_len else "fact_count_mismatch",
    }


def evaluate_page(pred_page: Dict[str, Any], true_page: Dict[str, Any]) -> Dict[str, Any]:
    """
    Evaluate a single page.

    Current page score is the mean of:
    - relaxed meta score
    - strict meta score
    - fact-count score

    Args:
        pred_page: Predicted page dictionary.
        true_page: Ground-truth page dictionary.

    Returns:
        Full page evaluation report.
    """
    meta_score = evaluate_meta(pred_page, true_page)
    facts_score = evaluate_facts(pred_page, true_page)

    page_score = mean([
        meta_score["soft_score"],
        meta_score["hard_score"],
        facts_score["facts_len_score"],
    ])

    return {
        "page_score": page_score,
        "meta_score": meta_score,
        "facts_score": facts_score,
    }


def build_page_debug_summary(page_report: Dict[str, Any]) -> Dict[str, Any]:
    """
    Build a compact debug summary for one evaluated page.

    Useful for scanning mistakes quickly without opening the full nested report.

    Args:
        page_report: Output of evaluate_page().

    Returns:
        Compact page debug summary.
    """
    meta = page_report["meta_score"]
    facts = page_report["facts_score"]

    meta_mismatches = []
    for field_name, info in meta["field_reports"].items():
        if not info["is_exact_match"]:
            meta_mismatches.append({
                "field": field_name,
                "method": info["method"],
                "hard_score": info["hard_score"],
                "soft_score": info["soft_score"],
                "used_score": info["used_score"],
                "pred_value": info["pred_value"],
                "true_value": info["true_value"],
            })

    return {
        "page_score": page_report["page_score"],
        "meta_hard_score": meta["hard_score"],
        "meta_soft_score": meta["soft_score"],
        "facts_len_score": facts["facts_len_score"],
        "meta_mismatches": meta_mismatches,
        "fact_count_debug": {
            "pred_fact_count": facts["pred_fact_count"],
            "true_fact_count": facts["true_fact_count"],
            "count_difference": facts["count_difference"],
            "is_exact_fact_count_match": facts["is_exact_fact_count_match"],
        },
    }


class Document:
    """
    Simple document container.

    Args:
        pages: List of page dictionaries.
    """

    def __init__(self, pages: List[Dict[str, Any]]):
        self.pages = pages


def evaluate_document(pred_doc: Document, true_doc: Document) -> Dict[str, Any]:
    """
    Evaluate a predicted document against a ground-truth document.

    Important:
    - Document length does not affect the score.
    - Only paired pages are evaluated.
    - `doc_len_match` is only a sanity flag.

    Args:
        pred_doc: Predicted document.
        true_doc: Ground-truth document.

    Returns:
        Document-level evaluation report.
    """
    pred_pages = pred_doc.pages
    true_pages = true_doc.pages

    doc_len_match = len(pred_pages) == len(true_pages)
    paired_count = min(len(pred_pages), len(true_pages))

    page_reports = []
    for i in range(paired_count):
        page_report = evaluate_page(pred_pages[i], true_pages[i])
        page_reports.append({
            "page_index": i,
            "page_score": page_report["page_score"],
            "report": page_report,
        })

    document_score = mean([p["page_score"] for p in page_reports]) if page_reports else 0.0

    return {
        "document_score": document_score,
        "doc_len_match": doc_len_match,
        "evaluated_page_count": paired_count,
        "page_reports": page_reports,
    }


def evaluate_document_detailed(pred_doc: Document, true_doc: Document) -> Dict[str, Any]:
    """
    Evaluate a document and return a more investigation-friendly report.

    Includes:
    - document_score
    - doc_len_match sanity flag
    - compact per-page debug summaries
    - full per-page reports

    Args:
        pred_doc: Predicted document.
        true_doc: Ground-truth document.

    Returns:
        Detailed document-level evaluation report.
    """
    pred_pages = pred_doc.pages
    true_pages = true_doc.pages

    doc_len_match = len(pred_pages) == len(true_pages)
    paired_count = min(len(pred_pages), len(true_pages))

    page_reports = []
    page_debug_summaries = []

    for i in range(paired_count):
        page_report = evaluate_page(pred_pages[i], true_pages[i])
        debug_summary = build_page_debug_summary(page_report)

        page_reports.append({
            "page_index": i,
            "page_score": page_report["page_score"],
            "report": page_report,
        })

        page_debug_summaries.append({
            "page_index": i,
            **debug_summary,
        })

    document_score = mean([p["page_score"] for p in page_reports]) if page_reports else 0.0

    return {
        "document_score": document_score,
        "doc_len_match": doc_len_match,
        "evaluated_page_count": paired_count,
        "page_debug_summaries": page_debug_summaries,
        "page_reports": page_reports,
    }


# Example usage:
#
pred_document = Document([pred_1_dict, pred_2_dict, pred_3_dict])
true_document = Document([gt_1, gt_2, gt_3])

report = evaluate_document(pred_document, true_document)
detailed_report = evaluate_document_detailed(pred_document, true_document)

NameError: name 'pred_1_dict' is not defined

In [ ]:
detailed_report["page_debug_summaries"]

In [ ]:
report['page_reports'][0]

In [20]:
import json
import pandas as pd

page_idx = 4

gt_path = "/Users/delmedigo/Dev/FineTree/data/annotations/P907062-00.json"
if page_idx < 10:
    pred_path = f"/Users/delmedigo/Dev/FineTree/benchmark/input/predictions/pred_000{page_idx}.json"
else:
    pred_path = f"/Users/delmedigo/Dev/FineTree/benchmark/input/predictions/pred_00{page_idx}.json"


def search_fact_by_num(_dict: dict, num: int):
    facts = _dict["facts"]
    candidates = [d for d in facts if d["fact_num"] == num]
    return candidates[0]

with open(gt_path, "r") as f:
    data = json.load(f)

gt = data.get("pages")[page_idx-1]

with open(pred_path, "r") as f:
    pred = json.loads(f.read())

for i,p in enumerate(pred["facts"]):
    p["fact_num"]=i+1

norm = lambda x: x.replace("״", '''"''').replace("(*)", "").strip()

def df_compare(by,pred=pred, gt=gt, method = "equality", path_idx_default=-1):
    if by == "path":
        df = pd.DataFrame(dict(p=[f[by][path_idx_default] for f in pred["facts"]], gt=[f[by][path_idx_default] for f in gt["facts"]]))
    else:
        df = pd.DataFrame(dict(p=[f[by] for f in pred["facts"]], gt=[f[by] for f in gt["facts"]]))
    if method == "equality":
        df["check"] = df.apply(lambda x: x["p"] == x["gt"], axis=1)
    elif method == "seq_matcher":
        df["check"] = df.apply(lambda x: char_similarity(a=norm(x["p"]), b=norm(x["gt"])), axis=1)
    return df


In [34]:
mesaurements = []
_by = "value"
_method = "equality"

for idx in range(1, 16):

    gt_path = "/Users/delmedigo/Dev/FineTree/data/annotations/P907062-00.json"
    if idx < 10:
        pred_path = f"/Users/delmedigo/Dev/FineTree/benchmark/input/predictions/pred_000{idx}.json"
    else:
        pred_path = f"/Users/delmedigo/Dev/FineTree/benchmark/input/predictions/pred_00{idx}.json"

    with open(gt_path, "r") as f:
        data = json.load(f)

    gt = data.get("pages")[idx-1]

    with open(pred_path, "r") as f:
        pred = json.loads(f.read())

    for i,p in enumerate(pred["facts"]):
        p["fact_num"]=i+1
        
    try:
        df_compared = df_compare(pred=pred, gt=gt,by=_by, method=_method)
        result = df_compared.check.mean()
        print(f"---comparing {idx} idx---")
        print(df_compared.head(10))
        print("---------------------\n")
        mesaurements.append(result)
    except Exception as e:
            print("error", e, "with", idx)
            if len(gt.get("facts")) ==0:
                pass
            else:              
                result = 0.0
                mesaurements.append(result)
    
    

print(mesaurements)
print(f"Final score for {_by} test:", sum(mesaurements) / len(mesaurements))

---comparing 1 idx---
Empty DataFrame
Columns: [p, gt, check]
Index: []
---------------------

---comparing 2 idx---
Empty DataFrame
Columns: [p, gt, check]
Index: []
---------------------

---comparing 3 idx---
Empty DataFrame
Columns: [p, gt, check]
Index: []
---------------------

---comparing 4 idx---
        p      gt  check
0       9       9   True
1      31      31   True
2       -       -   True
3       8       8   True
4       9       9   True
5      39      39   True
6    1793    1793   True
7    1795    1795   True
8  (1784)  (1784)   True
9  (1756)  (1756)   True
---------------------

error All arrays must be of the same length with 5
---comparing 6 idx---
        p      gt  check
0     121     121   True
1     574     574   True
2     695     695   True
3  (1570)  (1570)   True
4       -    (*)-  False
5  (1570)  (1570)   True
6  (1449)  (1449)   True
7     574     574   True
8   (875)   (875)   True
9   (881)   (881)   True
---------------------

---comparing 7 idx---
  

dict_keys(['image', 'meta', 'facts'])